In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report

In [2]:
df = pd.read_csv(r'C:\Infoct_project 1 sample practice\dataset\predictive_maintenance.csv')
print("Dataset loaded!")
print(df.shape)

Dataset loaded!
(10000, 10)


In [3]:
# Week 2 external context
np.random.seed(42)
df['Ambient_temperature'] = np.random.normal(loc=295, scale=3, size=len(df))
df['Load_density'] = np.random.uniform(low=0.3, high=1.0, size=len(df))

# Week 1 engineered features
df["temp_difference"] = df["Process temperature [K]"] - df["Air temperature [K]"]
df["power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]
df["tool_wear_rate"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"]
df["heat_stress_index"] = df["Air temperature [K]"] * df["Tool wear [min]"]
df["wear_per_rotation"] = df["Tool wear [min]"] / df["Rotational speed [rpm]"] * 1000

# Week 3 Day 1 new features
df["thermal_stress"] = df["temp_difference"] * df["tool_wear_rate"]
df["power_per_temp"] = df["power"] / df["Air temperature [K]"]

print("All features recreated!")
print(df.shape)

All features recreated!
(10000, 19)


In [4]:
# Final 12 features
final_features = ['Air temperature [K]', 'Process temperature [K]','Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]','temp_difference', 'power', 'tool_wear_rate',
'heat_stress_index', 'wear_per_rotation','thermal_stress', 'power_per_temp']
print("=== Feature Selection Summary ===")
print(f"\nTotal original columns: 10")
print(f"Total engineered features created: 11")
print(f"Features removed (weak): 4")
print(f"Final features selected: {len(final_features)}")
print(f"\nFinal 12 features:")
for i, feature in enumerate(final_features, 1):
    print(f"{i}. {feature}")

=== Feature Selection Summary ===

Total original columns: 10
Total engineered features created: 11
Features removed (weak): 4
Final features selected: 12

Final 12 features:
1. Air temperature [K]
2. Process temperature [K]
3. Rotational speed [rpm]
4. Torque [Nm]
5. Tool wear [min]
6. temp_difference
7. power
8. tool_wear_rate
9. heat_stress_index
10. wear_per_rotation
11. thermal_stress
12. power_per_temp


In [5]:
X = df[final_features]
Y = df['Target']
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y)
# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Train LightGBM
model = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42,
    class_weight='balanced'
)
model.fit(X_train_scaled, Y_train)
pred = model.predict(X_test_scaled)
score = f1_score(Y_test, pred, average='macro')
# Feature Importance Report
feature_importance = pd.DataFrame({
    'rank': range(1, len(final_features)+1),
    'feature': final_features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False).reset_index(drop=True)

feature_importance['rank'] = range(1, len(final_features)+1)

print("=== Feature Importance Report ===")
print(feature_importance[['rank', 'feature', 'importance']])
print(f"\nLightGBM Macro F1 Score: {score:.3f}")

[LightGBM] [Info] Number of positive: 271, number of negative: 7729
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000602 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
=== Feature Importance Report ===
    rank                  feature  importance
0      1   Rotational speed [rpm]         392
1      2                    power         380
2      3              Torque [Nm]         352
3      4          Tool wear [min]         312
4      5          temp_difference         297
5      6  Process temperature [K]         274
6      7           power_per_temp         254
7      8      Air temperature [K]         22

c:\Users\acer\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [6]:
print("=" * 50)
print("    WEEK 3 FINAL REPORT SUMMARY")
print("=" * 50)

print("\n DATASET:")
print(f"   Total rows: 10,000")
print(f"   Total columns: 10")
print(f"   Failure rate: 3.4% (339 machines)")

print("\n FEATURE ENGINEERING:")
print(f"   Rolling features (Week 1): 15")
print(f"   Engineered features created: 11")
print(f"   Features removed (weak): 4")
print(f"   Final features selected: 12")

print("\n MODEL PERFORMANCE:")
print(f"   Logistic Regression (Week 2): 0.693")
print(f"   LightGBM (Week 3): {score:.3f}")
print(f"   Improvement: {score - 0.693:.3f}")
print(f"   LightGBM is {((score-0.693)/0.693*100):.1f}% better!")

print("\n TOP 3 IMPORTANT FEATURES:")
print(f"   1. Rotational speed (392)")
print(f"   2. power (380)")
print(f"   3. Torque (352)")

print("\n KEY INSIGHTS:")
print(f"   Engineered features contribute 48% of model decisions!")
print(f"   LightGBM Macro F1: 0.899 - Production ready!")
print("=" * 50)

    WEEK 3 FINAL REPORT SUMMARY

 DATASET:
   Total rows: 10,000
   Total columns: 10
   Failure rate: 3.4% (339 machines)

 FEATURE ENGINEERING:
   Rolling features (Week 1): 15
   Engineered features created: 11
   Features removed (weak): 4
   Final features selected: 12

 MODEL PERFORMANCE:
   Logistic Regression (Week 2): 0.693
   LightGBM (Week 3): 0.899
   Improvement: 0.206
   LightGBM is 29.8% better!

 TOP 3 IMPORTANT FEATURES:
   1. Rotational speed (392)
   2. power (380)
   3. Torque (352)

 KEY INSIGHTS:
   Engineered features contribute 48% of model decisions!
   LightGBM Macro F1: 0.899 - Production ready!


In [7]:
print("=== Week 3 Review ===")
print()
print("Day 1 → Fusion Feature Engineering Review ")
print("       3 new features created!")
print()
print("Day 2 → Feature Selection & Optimization ")
print("       12 final features selected!")
print()
print("Day 3 → LightGBM Development ")
print("       Macro F1: 0.899 achieved!")
print()
print("Day 4 → Feature Importance & Explainability ")
print("       Engineered features = 48% importance!")
print()
print("Day 5 → Final Report ")
print("       All tasks completed!")
print()
print("Week 3 Complete! ")

=== Week 3 Review ===

Day 1 → Fusion Feature Engineering Review 
       3 new features created!

Day 2 → Feature Selection & Optimization 
       12 final features selected!

Day 3 → LightGBM Development 
       Macro F1: 0.899 achieved!

Day 4 → Feature Importance & Explainability 
       Engineered features = 48% importance!

Day 5 → Final Report 
       All tasks completed!

Week 3 Complete! 
